## 필요성
엑셀의 수식/서식/메모를 변경하지 않고 셀 값들만 변경할려면 엑셀객체에 api 접근하는 방법이 이용된다 합니다.
가용한 방법들은 아래와 같습니다.

+----------------------+-------------------+---------------------+------------------------------------------------+
| 구분                 | 라이브러리 / 기술 | 지원 운영체제       | 설명                                           |
+======================+===================+=====================+================================================+
| **VBA**              | 기본 매크로       | **Windows / macOS** | Excel 내장 스크립트 언어.                      |
|                      |                   |                     |                                                |
|                      |                   |                     | Excel 앱에서 직접 실행되며 모든 요소 완전 보존 |
|                      |                   |                     |                                                |
|                      |                   |                     | macOS에서 일부기능 제한적                      |
+----------------------+-------------------+---------------------+------------------------------------------------+
| **Python + pywin32** | `win32com.client` | **Windows 전용**    | Excel COM API 직접 호출.                       |
|                      |                   |                     |                                                |
|                      |                   |                     | VBA와 동일 수준의 제어 가능                    |
+----------------------+-------------------+---------------------+------------------------------------------------+
| **Python + xlwings** | `xlwings`         | **Windows / macOS** | COM(Windows)과 AppleScript(macOS) 기반의       |
|                      |                   |                     |                                                |
|                      |                   |                     | 고수준 래퍼.                                   |
|                      |                   |                     |                                                |
|                      |                   |                     | VBA 없이 Excel 완전 제어                       |
+----------------------+-------------------+---------------------+------------------------------------------------+
| **R + RDCOMClient**  | `RDCOMClient`     | **Windows 전용**    | R에서 Excel COM 객체를 직접 제어.              |
|                      |                   |                     |                                                |
|                      |                   |                     | VBA 대체 가능                                  |
+----------------------+-------------------+---------------------+------------------------------------------------+

이 방법들 중에서 Windows와 macOS에서 모두 가능한 방법을 선택해서 아래의 실습을 진행하코자 합니다.

In [1]:
import xlwings as xw

In [2]:
# 엑셀파일을 다루기 위해 인스턴스를 생성 = 실제로 엑셀파일이 열려짐
wb_target = xw.Book('deid_KQIPS eCRF (수신-전산팀) 20250508 수정_수술전후검사결과제공_20250529.xlsx')

In [3]:
## 엑셀파일 인스턴스로부터 시트 인스턴스를 얻는 3가지 방법
#모든 시트를 포함하는 컬랙션  
print(len(wb_target.sheets))
#시트명으로 조회
sheet_P_IC = wb_target.sheets['P_IC']
print(sheet_P_IC.name)
#시트순서로 조회  
sheet_1st = wb_target.sheets[0]
print(sheet_1st.name)

61
P_IC
P_IC


순회하면서 모든 시트를 탐색하도록 합니다.

In [4]:
i=0
for sheet in wb_target.sheets:
    i=i+1
    print(f"{i}. {sheet.name}")

1. P_IC
2. P_PRE_OP
3. P_PRE_OP;Q_SYMCATCO
4. P_PRE_OP;Q_SHABCATCO
5. P_PRE_OP;Q_SHNYNCO
6. P_PRE_OP;Q_MHETCYNCO
7. P_PRE_OP;Q_TFTRTCATCO
8. P_PRE_TEST
9. P_PRE_TEST;Q_LB
10. P_PRE_TEST;Q_LBE
11. P_PRE_TEST;Q_COLCATCO
12. P_PRE_TEST;Q_ANTRGCATCO
13. P_PRE_TEST;Q_ANIMMCATCO
14. P_OP
15. P_OP;Q_DXOPCATCO
16. P_OP;Q_TRTOCATCO
17. P_OP;Q_FDANCAT3
18. P_OP;Q_FDCOPCATCO
19. P_POST_TEST
20. P_POST_TEST;Q_LBA1
21. P_POST_TEST;Q_LBA2
22. P_POST_TEST;Q_LBA3
23. P_COMP
24. P_COMP;Q_CPR
25. P_COMP;Q_CPRC
26. P_COMP;Q_CPC
27. P_COMP;Q_CPCC
28. P_COMP;Q_CPL
29. P_COMP;Q_CPLC
30. P_COMP;Q_CPK
31. P_COMP;Q_CPKC
32. P_COMP;Q_CPM
33. P_COMP;Q_CPMC
34. P_COMP;Q_CPG
35. P_COMP;Q_CPGC
36. P_COMP;Q_CPI
37. P_COMP;Q_CPIC
38. P_COMP;Q_CPN
39. P_COMP;Q_CPNC
40. P_COMP;Q_CPP
41. P_COMP;Q_CPPC
42. P_COMP;Q_CPD
43. P_COMP;Q_CPDC
44. P_COMP;Q_CPO
45. P_COMP;Q_CPOC
46. P_COMP;Q_CPA
47. P_COMP;Q_CPAC
48. P_POST_CARE
49. P_POST_CARE;Q_ERCATCO
50. P_POST_CARE;Q_ERNPCATCO
51. P_POST_PATH
52. P_POST_PATH;Q_PATNM
53. P_P

In [5]:
import pandas as pd

In [6]:
wb_mapping = xw.Book('deid_전체환자_등록번호+등재번호.xlsx')
sheet_mapping = wb_mapping.sheets['mapping']
df = sheet_mapping.range('A1:B1').expand('down').options(pd.DataFrame, header=1, index=False).value
print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1857 entries, 0 to 1856
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   pseudo_등록번호  1857 non-null   object
 1   등재번호         1857 non-null   object
dtypes: object(2)
memory usage: 29.1+ KB
None


In [7]:
print(df['pseudo_등록번호'].is_unique)

False


In [8]:
df[df['pseudo_등록번호'].duplicated()]['pseudo_등록번호']

1418    37303444
Name: pseudo_등록번호, dtype: object

In [9]:
# 제거하기
sheet_mapping.api.Rows(1419).Delete()

True

In [10]:
wb_mapping.save('중복제거_deid_전체환자_등록번호+등재번호.xlsx')

com_error: (-2147352567, '예외가 발생했습니다.', (0, 'Microsoft Excel', '이 통합 문서를 이미 열려 있는 다른 통합 문서 또는 추가 기능과 같은 이름으로 저장할 수 없습니다. 다른 이름을 사용하거나 열려 있는 다른 파일을 닫은 후 저장하십시오.', 'xlmain11.chm', 0, -2146827284), None)

In [ ]:
wb_mapping.close() 

In [ ]:
wb_mapping = xw.Book('중복제거_deid_전체환자_등록번호+등재번호.xlsx')
sheet_mapping = wb_mapping.sheets['mapping']
df = sheet_mapping.range('A1:B1').expand('down').options(pd.DataFrame, header=1, index=False).value
print(df.info())

In [ ]:
sheet_1st.range('B8').value = 'VLOOKUP'

In [ ]:
sheet_1st.range('B8').formula = '=VLOOKUP(A8, [중복제거_deid_전체환자_등록번호+등재번호.xlsx]mapping!A:B, 2, FALSE)'